# 🔥 Bloque 2: Deep Learning con PyTorch

**Objetivo:** Entender tensores, redes neuronales, entrenamiento y las arquitecturas fundamentales (MLP, CNN).

---

## 1. ¿Por qué PyTorch?

PyTorch es el framework de Deep Learning más usado en investigación y cada vez más en producción. Sus ventajas:
- **Pythónico**: siente como numpy pero con autograd
- **Dynamic graphs**: define el grafo de computación en tiempo real
- **Ecosistema**: HuggingFace, Lightning, Pyro... todo se basa en PyTorch

---

## 2. Tensores — el bloque básico

In [ ]:
# !pip install torch torchvision matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando: {device}")

In [ ]:
# Crear tensores
t1 = torch.tensor([1.0, 2.0, 3.0])          # 1D (vector)
t2 = torch.zeros(3, 4)                        # Matriz 3x4 de ceros
t3 = torch.randn(2, 3, 4)                     # Tensor 3D aleatorio (normal)

print(f"t1 shape: {t1.shape}, dtype: {t1.dtype}")
print(f"t2 shape: {t2.shape}")
print(f"t3 shape: {t3.shape}")

# Operaciones
a = torch.tensor([[1., 2.], [3., 4.]])
b = torch.tensor([[5., 6.], [7., 8.]])

print(f"\nSuma:\n{a + b}")
print(f"\nMatmul:\n{torch.matmul(a, b)}")
print(f"\nMedia: {a.mean()}, Std: {a.std()}")

## 3. Autograd — gradientes automáticos

> Esto es lo que hace especial a PyTorch. Calcula derivadas automáticamente para el backpropagation.

In [ ]:
# requires_grad=True le dice a PyTorch que rastree operaciones sobre este tensor
x = torch.tensor(3.0, requires_grad=True)

# Operación: y = x^2 + 2x + 1
y = x**2 + 2*x + 1

# Calcular gradiente: dy/dx = 2x + 2 = 2(3) + 2 = 8
y.backward()

print(f"y = {y.item():.1f}")
print(f"dy/dx = {x.grad.item():.1f}  (esperado: 8.0)")

## 4. Red Neuronal desde cero (MLP)

Vamos a construir un **Multilayer Perceptron** para clasificar el dataset Iris.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader

# Preparar datos
iris = load_iris()
X, y = iris.data, iris.target

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# Convertir a tensores PyTorch
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.LongTensor(y_train)   # LongTensor para clasificación
X_test_t  = torch.FloatTensor(X_test)
y_test_t  = torch.LongTensor(y_test)

# DataLoader: gestiona batches automáticamente
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader  = DataLoader(train_dataset, batch_size=16, shuffle=True)

print(f"Batches por epoch: {len(train_loader)}")

In [ ]:
# Definir la red neuronal
class MLP(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(MLP, self).__init__()
        
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size),   # Capa oculta 1
            nn.ReLU(),                             # Activación no lineal
            nn.Dropout(0.2),                       # Regularización
            nn.Linear(hidden_size, hidden_size),   # Capa oculta 2
            nn.ReLU(),
            nn.Linear(hidden_size, num_classes)    # Capa de salida
        )
    
    def forward(self, x):
        return self.network(x)

# Instanciar modelo
model = MLP(input_size=4, hidden_size=32, num_classes=3).to(device)
print(model)
print(f"\nParámetros entrenables: {sum(p.numel() for p in model.parameters()):,}")

## 5. Loop de entrenamiento

> El loop de entrenamiento siempre tiene la misma estructura en PyTorch:
> 1. Forward pass → calcular predicción
> 2. Calcular loss
> 3. Backward pass → calcular gradientes
> 4. Optimizer step → actualizar pesos

In [ ]:
# Loss y optimizador
criterion = nn.CrossEntropyLoss()                       # Para clasificación multiclase
optimizer = optim.Adam(model.parameters(), lr=0.001)    # Adam es el más usado

# Entrenamiento
EPOCHS = 100
train_losses = []

for epoch in range(EPOCHS):
    model.train()          # Modo entrenamiento (activa dropout, etc.)
    epoch_loss = 0
    
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()           # 1. Limpiar gradientes anteriores
        outputs = model(X_batch)        # 2. Forward pass
        loss = criterion(outputs, y_batch)  # 3. Calcular loss
        loss.backward()                 # 4. Backward pass
        optimizer.step()                # 5. Actualizar pesos
        
        epoch_loss += loss.item()
    
    train_losses.append(epoch_loss / len(train_loader))
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {train_losses[-1]:.4f}")

In [ ]:
# Visualizar curva de loss
plt.figure(figsize=(8, 4))
plt.plot(train_losses, color='steelblue')
plt.title('Curva de Loss durante entrenamiento')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluación
model.eval()   # Modo evaluación (desactiva dropout)

with torch.no_grad():   # No calcular gradientes (ahorra memoria y tiempo)
    X_test_device = X_test_t.to(device)
    outputs = model(X_test_device)
    _, predicted = torch.max(outputs, 1)   # Clase con mayor probabilidad
    predicted = predicted.cpu().numpy()

from sklearn.metrics import classification_report
print(classification_report(y_test, predicted, target_names=iris.target_names))

## 6. Conceptos clave para recordar

| Concepto | Qué hace |
|---|---|
| `nn.Linear(in, out)` | Capa fully-connected: y = Wx + b |
| `nn.ReLU()` | Activación: max(0, x). Introduce no-linealidad |
| `nn.Dropout(p)` | Desactiva neuronas aleatoriamente durante train (evita overfitting) |
| `CrossEntropyLoss` | Loss para clasificación multiclase |
| `Adam` | Optimizador adaptativo. El más usado en práctica |
| `model.train()` | Activa dropout y batch norm en modo entrenamiento |
| `model.eval()` | Desactiva dropout para inferencia |
| `torch.no_grad()` | No computar gradientes (inferencia más rápida) |

---

## ✅ Resumen del bloque

- Entiendes qué son **tensores** y cómo operarlos
- Comprendes **autograd** y por qué es clave en DL
- Sabes construir una red neuronal con `nn.Module`
- Implementaste el **loop de entrenamiento** completo en PyTorch
- Conoces la diferencia entre `model.train()` y `model.eval()`

---

## ➡️ Siguiente paso

Continúa con el **Bloque 3: Modelos Probabilísticos y Bayesianos** → `03_probabilistic_models.ipynb`